# QAOA – universal template

Workflow: **problem → QUBO → Ising Hamiltonian → QAOA circuit → parameter optimization → sampling → evaluation**.

**How to use it**
1. Everything is set in the **⚙️ Configuration** cell. The other cells don't need to be changed.
2. The execution mode is chosen with the `MODE` variable:

   | `MODE` | what happens | IBM connection | quota |
   |---|---|---|---|
   | `"local"` | **simulation only**: exact, noiseless (Statevector), up to ~24 qubits | no | not used |
   | `"fake"` | **simulation only**: with the noise of a real processor (`FAKE_BACKEND`, default FakeMarrakesh) | no | not used |
   | `"hardware"` | a real IBM quantum computer | yes | used |

   The recommended order is `"local"` → `"fake"` → `"hardware"`. In `local` and `fake` modes the whole notebook runs without an IBM account and without internet.
3. For a problem other than Max-Cut, edit the `build_problem` function in section **2. Problem**.

**Safeguards against wasting quota**
- Nothing is sent to hardware until you set `RUN_ON_HARDWARE = True`.
- If the transpiled circuit has more than `MAX_2Q_GATES` two-qubit gates, the run stops. Such a circuit would return only noise on hardware.
- `MAXITER` limits the number of iterations and `MAX_TIME` the length of the session.
- ⚠️ **A session is billed for its entire duration**, including the time spent on classical computation between jobs. The `"job"` or `"batch"` mode bills only QPU time, but each job may wait in the queue.
- Large circuits are not drawn. A picture of a 100-qubit circuit was over 200 MB.

In [ ]:
import time
from contextlib import nullcontext

import matplotlib.pyplot as plt
import numpy as np
import rustworkx as rx
from IPython.display import display
from rustworkx.visualization import mpl_draw as draw_graph
from scipy.optimize import minimize

from qiskit.circuit.library import QAOAAnsatz
from qiskit.primitives import StatevectorEstimator, StatevectorSampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import Batch, QiskitRuntimeService, Session, fake_provider
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime import SamplerV2 as Sampler

from qiskit_addon_opt_mapper.applications import Maxcut, Tsp, VehicleRouting
from qiskit_addon_opt_mapper.converters import OptimizationProblemToQubo
from qiskit_addon_opt_mapper.translators import to_ising

## ⚙️ Configuration

In [ ]:
# ── Graph ─────────────────────────────────────────────────────────────────────
GRAPH_KIND   = "regular"   # "regular" | "random" | "ring" | "complete" | "native" | "manual"
N_NODES      = 12          # number of nodes (= number of qubits for Max-Cut)
DEGREE       = 3           # "regular": every node has exactly DEGREE neighbors
EDGE_PROB    = 0.3         # "random":  probability of an edge between each pair of nodes
WEIGHTED     = False       # True = random edge weights from WEIGHT_RANGE, otherwise all 1.0
WEIGHT_RANGE = (0.1, 1.0)
MANUAL_EDGES = [(0, 1, 1.0), (1, 2, 1.0), (2, 3, 1.0), (3, 0, 1.0), (0, 2, 1.0)]  # "manual": (i, j, weight)
SEED         = 42          # same SEED = same graph
PROBLEM      = "
"     # "maxcut" | "tsp" | "vrp"

# ── QAOA and optimization ─────────────────────────────────────────────────────
REPS    = 2       # p = number of QAOA layers
MAXITER = 50      # max. COBYLA iterations (= max. number of Estimator jobs)
RHOBEG  = 0.3     # initial COBYLA step size
TOL     = 1e-3

# ── Where to run ──────────────────────────────────────────────────────────────
MODE         = "local"          # "local"    = simulation only, exact, noiseless   (no IBM, free)
                                # "fake"     = simulation only, FAKE_BACKEND noise (no IBM, free)
                                # "hardware" = real IBM quantum computer           (uses quota)
BACKEND_NAME = None             # e.g. "ibm_marrakesh"; None = least busy processor
FAKE_BACKEND = "FakeMarrakesh"  # noise model for MODE="fake" (and topology for GRAPH_KIND="native")
EXECUTION    = "session"        # hardware only: "session" | "batch" | "job"
MAX_TIME     = "15m"            # max session/batch length – the whole duration is billed!
OPT_LEVEL    = 3                # transpiler optimization level

ESTIMATOR_SHOTS = 1_000
SAMPLER_SHOTS   = 10_000
JOB_TAGS        = ["QAOA_TEMPLATE"]

# ── Safeguards ────────────────────────────────────────────────────────────────
MAX_LOCAL_QUBITS = 24      # exact simulation needs ~2^n memory
MAX_2Q_GATES     = 2_000   # more two-qubit gates = practically pure noise on hardware
RUN_ON_HARDWARE  = False   # unless True, nothing is sent to real hardware

## 0. Backend

In [ ]:
service = None
if MODE == "hardware":
    service = QiskitRuntimeService()
    backend = (service.backend(BACKEND_NAME) if BACKEND_NAME
               else service.least_busy(operational=True, simulator=False, min_num_qubits=N_NODES))
    print("Instance:", service.active_account().get("instance"))
elif MODE == "fake":
    backend = getattr(fake_provider, FAKE_BACKEND)()
elif MODE == "local":
    backend = None
else:
    raise ValueError(f"Neznámý MODE: {MODE!r}")

# processor topology – needed for GRAPH_KIND="native" (FAKE_BACKEND is used in local mode)
topology = backend if backend is not None else getattr(fake_provider, FAKE_BACKEND)()
print("Režim:", MODE, "| backend:", backend.name if backend else "StatevectorEstimator (bez šumu)")
if MODE != "hardware":
    print("🖥  Jen simulace – žádné připojení k IBM, kvóta se nečerpá.")

## 1. Graph – automatic edge generation

| `GRAPH_KIND` | edges | suitable for hardware |
|---|---|---|
| `"regular"` | every node has exactly `DEGREE` neighbors (classic QAOA benchmark) | ✅ for small `DEGREE` |
| `"random"` | each pair of nodes with probability `EDGE_PROB` | ⚠️ only for small `EDGE_PROB` |
| `"ring"` | ring 0–1–2–…–(n−1)–0 | ✅ |
| `"complete"` | every node with every other | ❌ many SWAPs, results are noise |
| `"native"` | exactly copies the processor's qubit connectivity | ✅✅ the circuit needs no SWAPs |
| `"manual"` | `MANUAL_EDGES` | – |

In [ ]:
def random_regular_edges(n, d, rng, max_tries=2_000):
    # Random d-regular graph: random pairing of "stubs", repeated until the graph is valid.
    if not (0 < d < n) or (n * d) % 2:
        raise ValueError("Pro d-regulární graf musí platit 0 < d < n a n·d sudé.")
    for _ in range(max_tries):
        stubs = rng.permutation(np.repeat(np.arange(n), d)).reshape(-1, 2)
        pairs = {tuple(sorted(map(int, p))) for p in stubs}
        if len(pairs) == len(stubs) and all(i != j for i, j in pairs):
            return sorted(pairs)
    raise RuntimeError("Nepodařilo se vygenerovat d-regulární graf – zkus jiný SEED nebo menší DEGREE.")


def native_edges(backend, n, start_qubit=0):
    # n connected qubits chosen by breadth-first search (BFS) over the processor's coupling map.
    # Returns the edges (renumbered to 0..n-1) and the list of physical qubits for initial_layout.
    cmap = backend.coupling_map.graph.to_undirected(multigraph=False)
    order, seen, k = [start_qubit], {start_qubit}, 0
    while len(order) < n and k < len(order):
        for nb in sorted(cmap.neighbors(order[k])):
            if nb not in seen and len(order) < n:
                seen.add(nb)
                order.append(nb)
        k += 1
    if len(order) < n:
        raise ValueError(f"Procesor {backend.name} nemá {n} propojených qubitů.")
    idx = {q: i for i, q in enumerate(order)}
    edges = sorted({tuple(sorted((idx[a], idx[b]))) for a, b in cmap.edge_list() if a in idx and b in idx})
    return edges, order


def make_edges(kind, n, rng):
    # Returns (list of edges, physical qubits or None).
    if kind == "regular":
        return random_regular_edges(n, DEGREE, rng), None
    if kind == "random":
        return [(i, j) for i in range(n) for j in range(i + 1, n) if rng.random() < EDGE_PROB], None
    if kind == "ring":
        return [(i, (i + 1) % n) for i in range(n)], None
    if kind == "complete":
        return [(i, j) for i in range(n) for j in range(i + 1, n)], None
    if kind == "native":
        return native_edges(topology, n)
    if kind == "manual":
        return MANUAL_EDGES, None
    raise ValueError(f"Neznámý GRAPH_KIND: {kind!r}")


rng = np.random.default_rng(SEED)
raw_edges, physical_qubits = make_edges(GRAPH_KIND, N_NODES, rng)
edges = [(int(e[0]), int(e[1]),
          float(e[2]) if len(e) == 3 else (round(float(rng.uniform(*WEIGHT_RANGE)), 3) if WEIGHTED else 1.0))
         for e in raw_edges]
n_nodes = 1 + max(max(i, j) for i, j, _ in edges) if GRAPH_KIND == "manual" else N_NODES

graph = rx.PyGraph()
graph.add_nodes_from(range(n_nodes))
graph.add_edges_from(edges)

degrees = [graph.degree(v) for v in graph.node_indices()]
print(f"Graf '{GRAPH_KIND}': {graph.num_nodes()} uzlů, {graph.num_edges()} hran, "
      f"stupeň {min(degrees)}–{max(degrees)}, souvislý: {rx.is_connected(graph)}")
if physical_qubits is not None:
    print("Fyzické qubity:", physical_qubits)

if n_nodes <= 60:
    plt.figure(figsize=(6, 6))
    draw_graph(graph, node_size=300 if n_nodes <= 30 else 60, with_labels=n_nodes <= 30,
               node_color="tab:blue", font_color="white",
               edge_labels=(lambda w: f"{w:g}") if WEIGHTED and n_nodes <= 20 else None)
    plt.show()

## 2. Problem → QUBO → Ising Hamiltonian

The `build_problem` function can return **any** `OptimizationProblem`: maximization or minimization, with or without constraints. `OptimizationProblemToQubo` converts constraints into penalties. Examples:

```python
from qiskit_addon_opt_mapper.applications import Knapsack, VertexCover, NumberPartition

def build_problem(graph):
    return Knapsack(values=[3, 4, 5, 6], weights=[2, 3, 4, 5], max_weight=7).to_optimization_problem()
    # return VertexCover(graph).to_optimization_problem()
    # return NumberPartition([3, 1, 1, 2, 2, 1]).to_optimization_problem()
```

For problems that don't live on a graph, set `PLOT_ON_GRAPH = False`.

In [ ]:
def build_problem(graph):
    if PROBLEM == "maxcut":
        return Maxcut(graph).to_optimization_problem()
    elif PROBLEM == "tsp":
        return Tsp(graph).to_optimization_problem()
    elif PROBLEM == "vrp":
        return VehicleRouting(graph).to_optimization_problem()

PLOT_ON_GRAPH = True   # color the resulting solution on the graph (variable i = node i)

problem   = build_problem(graph)
converter = OptimizationProblemToQubo()
qubo      = converter.convert(problem)
cost_hamiltonian, offset = to_ising(qubo)
n_qubits  = cost_hamiltonian.num_qubits
MAXIMIZE  = problem.objective.sense.name == "MAXIMIZE"


def objective_of(x):
    # Value of the original objective and feasibility for bits x (variable i = qubit i).
    xs = converter.interpret(np.asarray(x))
    return float(problem.objective.evaluate(xs)), bool(problem.is_feasible(xs))


def energy_to_objective(e):
    # Ising Hamiltonian energy → value of the original objective (including penalties).
    q = e + offset
    return -q if MAXIMIZE else q


# Reference: random guessing (uniform distribution) and exact brute-force optimum (small n only)
E_uniform = float(sum(c.real for p, c in zip(cost_hamiltonian.paulis, cost_hamiltonian.coeffs)
                      if not (p.z.any() or p.x.any())))
E_opt = x_opt = None
if n_qubits <= 20:
    diag = np.real(cost_hamiltonian.to_matrix(sparse=True).diagonal())
    k = int(np.argmin(diag))
    E_opt, x_opt = float(diag[k]), [(k >> i) & 1 for i in range(n_qubits)]

print(f"Problém: {'maximalizace' if MAXIMIZE else 'minimalizace'}, {problem.get_num_vars()} proměnných "
      f"→ {n_qubits} qubitů, {len(cost_hamiltonian)} Pauliho členů, offset = {offset:g}")
print(f"Náhodné hádání:  objektiv ≈ {energy_to_objective(E_uniform):g}")
if E_opt is not None:
    print(f"Přesné optimum:  objektiv = {energy_to_objective(E_opt):g}  (hrubou silou)")

if MODE == "local" and n_qubits > MAX_LOCAL_QUBITS:
    raise RuntimeError(f"{n_qubits} qubitů je na přesnou simulaci moc (limit {MAX_LOCAL_QUBITS}). "
                       "Zmenši problém nebo použij MODE='fake'/'hardware'.")

## 3. QAOA circuit and transpilation

After transpilation, check the **number of two-qubit gates**. With an error rate of roughly 10⁻³ per gate, thousands of gates are still usable, but tens of thousands return only noise.

In [ ]:
ansatz = QAOAAnsatz(cost_operator=cost_hamiltonian, reps=REPS)
ansatz.measure_all()

if MODE == "local":
    pm = generate_preset_pass_manager(optimization_level=1, basis_gates=["rz", "sx", "x", "cx"])
else:
    use_native = physical_qubits is not None and len(physical_qubits) == n_qubits
    pm = generate_preset_pass_manager(optimization_level=OPT_LEVEL, backend=backend,
                                      initial_layout=physical_qubits if use_native else None,
                                      seed_transpiler=SEED)

t0 = time.perf_counter()
isa_sampler     = pm.run(ansatz)                                         # with measurement → Sampler
isa_estimator   = isa_sampler.remove_final_measurements(inplace=False)   # without measurement → Estimator
isa_hamiltonian = cost_hamiltonian.apply_layout(isa_estimator.layout)

n_2q  = sum(1 for inst in isa_estimator.data
            if inst.operation.num_qubits == 2 and inst.operation.name != "barrier")
depth = isa_estimator.depth(lambda inst: inst.operation.num_qubits == 2)
print(f"Transpilace {time.perf_counter() - t0:.1f} s | dvouqubitových bran: {n_2q} | "
      f"2q hloubka: {depth} | celková hloubka: {isa_estimator.depth()}")

if MODE != "local" and n_2q > MAX_2Q_GATES:
    msg = (f"Obvod má {n_2q} dvouqubitových bran (limit MAX_2Q_GATES = {MAX_2Q_GATES}). Výsledek by byl šum – "
           "zmenši graf, sniž hustotu hran, REPS, nebo použij GRAPH_KIND='native'.")
    if MODE == "hardware":
        raise RuntimeError(msg)
    print("⚠️", msg)

if ansatz.num_qubits <= 16:
    display(ansatz.draw("mpl"))   # large circuits are deliberately not drawn (images can reach hundreds of MB)

## 4. Parameter optimization (COBYLA)

The initial point is a linear "ramp": γ gradually increases and β decreases, similar to an adiabatic transition. On hardware, an estimate of the total time is printed after the first iteration.

In [ ]:
def initial_point(ansatz, reps, delta=0.6):
    # Values in the same order as ansatz.parameters (β[0..p-1], γ[0..p-1]).
    gamma_scale = 0.5 / max(abs(c) for c in cost_hamiltonian.coeffs)   # scaled by the size of the coefficients
    x0 = []
    for prm in ansatz.parameters:
        t = (prm.index + 0.5) / reps
        x0.append(delta * (1 - t) if prm.vector.name == "β" else delta * t * gamma_scale)
    return np.array(x0)


def configure(prim):
    # Error suppression and job tags – only meaningful on real hardware.
    if MODE == "hardware":
        prim.options.environment.job_tags = JOB_TAGS
        prim.options.dynamical_decoupling.enable = True
        prim.options.dynamical_decoupling.sequence_type = "XY4"
        prim.options.twirling.enable_gates = True
        prim.options.twirling.num_randomizations = "auto"
    return prim


def make_estimator(mode):
    if MODE == "local":
        return StatevectorEstimator()
    est = Estimator(mode=mode)
    est.options.default_shots = ESTIMATOR_SHOTS
    return configure(est)


def execution_context():
    if MODE == "hardware" and not RUN_ON_HARDWARE:
        raise RuntimeError("Pojistka: na hardware se nic neodešle, dokud nenastavíš RUN_ON_HARDWARE = True.")
    if MODE != "hardware" or EXECUTION == "job":
        return nullcontext(backend)
    return {"session": Session, "batch": Batch}[EXECUTION](backend=backend, max_time=MAX_TIME)


def fmt_params(values):
    return {p.name: round(float(v), 4) for p, v in zip(ansatz.parameters, values)}


def cost(params, estimator):
    t = time.perf_counter()
    res = estimator.run([(isa_estimator, isa_hamiltonian, params)]).result()[0]
    e, s = float(np.ravel(res.data.evs)[0]), float(np.ravel(res.data.stds)[0])
    history.append({"energy": e, "std": s, "params": np.array(params), "seconds": time.perf_counter() - t})
    k = len(history)
    print(f"iter {k:3d}  ⟨H⟩ = {e:+.4f} ± {s:.4f}  → objektiv ≈ {energy_to_objective(e):.4g}"
          f"  ({history[-1]['seconds']:.1f} s)")
    if k == 1 and MODE == "hardware":
        sec = history[0]["seconds"]
        print(f"   ⏱ odhad: až {MAXITER} × {sec:.0f} s ≈ {MAXITER * sec / 60:.0f} min "
              f"(strop session/batch: {MAX_TIME})")
    return e

In [ ]:
history = []
x0 = initial_point(ansatz, REPS)
print("Počáteční parametry:", fmt_params(x0))

t_start, interrupted, result, ctx = time.perf_counter(), False, None, execution_context()
with ctx as mode:
    estimator = make_estimator(mode)
    try:
        result = minimize(cost, x0, args=(estimator,), method="COBYLA", tol=TOL,
                          options={"maxiter": MAXITER, "rhobeg": RHOBEG})
    except KeyboardInterrupt:
        interrupted = True
        print("⏹ Přerušeno – pokračuji s nejlepšími dosud naměřenými parametry.")

best_params = result.x if result is not None else min(history, key=lambda h: h["energy"])["params"]
print(f"\nHotovo: {len(history)} iterací za {(time.perf_counter() - t_start) / 60:.1f} min"
      + (f" | {result.message}" if result is not None else ""))
print("Optimální parametry:", fmt_params(best_params))
if isinstance(ctx, (Session, Batch)):
    try:
        print(f"Účtovaná doba {EXECUTION}: {ctx.usage()} s")
    except Exception as exc:
        print("Účtovanou dobu zatím nelze zjistit:", exc)

In [ ]:
energies = np.array([h["energy"] for h in history])
stds     = np.array([h["std"] for h in history])
it       = np.arange(1, len(history) + 1)

plt.figure(figsize=(10, 4.5))
plt.plot(it, energies, "o-", ms=3, label="⟨H⟩")
if stds.any():
    plt.fill_between(it, energies - stds, energies + stds, alpha=0.2, label="± std")
plt.axhline(E_uniform, ls="--", c="gray", label="náhodné hádání")
if E_opt is not None:
    plt.axhline(E_opt, ls=":", c="tab:green", label="optimum")
plt.xlabel("Iterace")
plt.ylabel("Energie ⟨H⟩  (nižší = lepší)")
plt.legend()
plt.show()

## 5. Sampling with the optimal parameters

On hardware, sampling is sent as **a single standalone job**, not within the session. So only its QPU time is billed.

In [ ]:
if MODE == "local":
    sampler = StatevectorSampler(default_shots=SAMPLER_SHOTS, seed=SEED)
else:
    if MODE == "hardware" and not RUN_ON_HARDWARE:
        raise RuntimeError("Pojistka: nastav RUN_ON_HARDWARE = True.")
    sampler = configure(Sampler(mode=backend))
    sampler.options.default_shots = SAMPLER_SHOTS

job = sampler.run([(isa_sampler, best_params)])
counts = job.result()[0].data.meas.get_counts()
shots = sum(counts.values())
print(f"{shots} vzorků, {len(counts)} různých bitových řetězců")
if MODE == "hardware":
    print("Job:", job.job_id(), "| využití:", job.usage(), "s")

## 6. Evaluation

In [ ]:
# Qiskit bitstrings are little-endian: last character = qubit 0 = variable 0
samples = []
for bs, c in counts.items():
    x = np.array([int(b) for b in reversed(bs)])
    val, feas = objective_of(x)
    samples.append({"bits": bs, "x": x, "p": c / shots, "value": val, "feasible": feas})

sign = -1 if MAXIMIZE else 1
feasible = [s for s in samples if s["feasible"]] or samples
best     = min(feasible, key=lambda s: sign * s["value"])
likely   = max(samples, key=lambda s: s["p"])
mean_val = sum(s["p"] * s["value"] for s in samples)
p_feas   = sum(s["p"] for s in samples if s["feasible"])

print(f"Nejlepší vzorek:      {best['value']:g}   (p = {best['p']:.4f}, přípustný: {best['feasible']})")
print(f"Nejčastější vzorek:   {likely['value']:g}   (p = {likely['p']:.4f})")
print(f"Střední hodnota:      {mean_val:g}")
print(f"Přípustné vzorky:     {p_feas:.1%}")
print(f"Náhodné hádání:       {energy_to_objective(E_uniform):g}")
if E_opt is not None:
    opt_val = objective_of(x_opt)[0]
    p_opt = sum(s["p"] for s in samples if s["feasible"] and np.isclose(s["value"], opt_val))
    print(f"Optimum:              {opt_val:g}   → nejlepší vzorek / optimum = "
          f"{best['value'] / opt_val if opt_val else float('nan'):.3f}, "
          f"P(trefit optimum) = {p_opt:.2%}")
if n_qubits <= 64:
    print("Nejlepší bity x[0..n-1]:", "".join(map(str, best["x"])))

In [ ]:
# distribution of objective values (weighted by probability)
vals = {}
for s in samples:
    vals[s["value"]] = vals.get(s["value"], 0) + s["p"]
xs = sorted(vals)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(len(xs)), [vals[v] for v in xs], color="tab:grey")
ax.set_xticks(range(len(xs)))
ax.set_xticklabels([f"{v:g}" for v in xs], rotation=45 if len(xs) > 15 else 0)
ax.set_xlabel("Hodnota účelové funkce")
ax.set_ylabel("Pravděpodobnost")
ax.set_title("Rozdělení výsledků QAOA")
plt.tight_layout()
plt.show()

# best solution on the graph (Max-Cut: two colors = two sides of the cut, red edges = cut edges)
if PLOT_ON_GRAPH and n_nodes <= 60 and len(best["x"]) >= n_nodes:
    colors = ["tab:purple" if best["x"][v] else "tab:orange" for v in graph.node_indices()]
    edge_colors = ["tab:red" if best["x"][a] != best["x"][b] else "lightgray" for a, b in graph.edge_list()]
    plt.figure(figsize=(6, 6))
    draw_graph(graph, node_color=colors, edge_color=edge_colors, with_labels=n_nodes <= 30,
               node_size=300 if n_nodes <= 30 else 60, font_color="white", width=2)
    plt.title(f"Nejlepší nalezené řešení: {best['value']:g}")
    plt.show()

In [ ]:
# remaining instance quota (rolling 28-day window)
if service is not None:
    u = service.usage()
    print(f"Spotřebováno: {u['usage_consumed_seconds'] / 60:.0f} min | "
          f"zbývá: {u['usage_remaining_seconds'] / 60:.0f} min z {u['usage_limit_seconds'] / 60:.0f} min")